# EEG raw (.fif, sau Layer 2) — xử lý bằng HAPPE (Python port) + báo cáo QC

Notebook này chạy **HAPPE** (Harvard Automated Processing Pipeline for EEG,
Gabard-Durnam et al. 2018, *Frontiers in Neuroscience* — v4.1, bản port
Python nằm ở `happe-python/`) trên toàn bộ file `eeg_labeled_raw.fif` do
Layer 2 xuất (2 kênh AF3_RAW/AF4_RAW, 244Hz, đã gắn annotation
REST_BEFORE/MEDITATION/REST_AFTER), rồi tổng hợp + trực quan hoá báo cáo QC
mà HAPPE xuất ra.

## HAPPE là gì, dùng để làm gì ở đây

HAPPE là pipeline tiền xử lý EEG chuẩn hoá, tự động, được thiết kế đặc biệt
cho dữ liệu "khó" (trẻ em, lâm sàng, ít kênh) — publication gốc benchmark
trên EEG trẻ sơ sinh/trẻ nhỏ. Các bước chính (xem `happe-python/README.md`
để biết chi tiết map bước → module):

1. Channel corrections → 2. Line-noise reduction (loại nhiễu điện lưới) →
3. Resample → 4. Band-pass → 5-6. Phát hiện kênh xấu → 7. ECGone (tắt, xem
dưới) → 8. **Wavelet thresholding** (bước làm sạch artefact CHÍNH, thay thế
ICA truyền thống — quan trọng vì dữ liệu ở đây chỉ có 2 kênh, ICA cần nhiều
kênh hơn để tách nguồn có ý nghĩa) → 9. MuscIL (tắt, xem dưới) → 12. Tính
chỉ số QC → 14. Cắt đoạn (segment) 2s → 17. Loại đoạn theo biên độ →
18. Nội suy kênh xấu → 19. Re-reference → 21. Trực quan hoá → 22-23. Lưu +
xuất báo cáo QC.

## Vì sao dùng cấu hình khác mặc định HAPPE ở vài chỗ — và giới hạn cần biết

- **`badChans.density = "low"`**: mức thấp nhất HAPPE hỗ trợ, nhưng vẫn được
  thiết kế cho lưới ≥32 kênh. Với CHỈ 2 kênh (AF3/AF4), phép z-score "so kênh
  này với các kênh khác" gần như suy biến (n=2) — chỉ tin cậy để bắt lỗi
  flatline rõ ràng (kênh chết hẳn), KHÔNG nên diễn giải "% kênh tốt" như một
  chỉ số tinh vi.
- **`muscIL.enabled = False`**: bước này dùng ICA để tách + loại thành phần
  cơ mặt — ICA với 2 kênh không tách nguồn có ý nghĩa (số thành phần ICA tối
  đa = số kênh = 2), nên tắt hẳn thay vì chạy 1 phép ICA gần như vô nghĩa.
  Việc làm sạch artefact chính dồn hết vào bước wavelet (bước 8).
- **`lineNoise.freq = 50.0`** (không phải mặc định 60Hz của Mỹ) — Việt Nam
  dùng điện lưới 50Hz. Kiểm tra bằng `lineNoise_coherence_50Hz`: coherence
  tụt rõ rệt tại 50Hz so với các tần số lân cận là dấu hiệu lọc thành công.
- **`reref` (re-reference)**: giữ mặc định HAPPE (`average`) nhưng
  **CHƯA THỂ XÁC NHẬN** thiết bị đã ghi dữ liệu tham chiếu kiểu gì ở tầng
  phần cứng (so 1 điện cực ẩn, hay vi sai AF3-AF4) — áp thêm average-reference
  phần mềm lên 2 kênh chỉ đơn giản trừ đi trung bình (AF3+AF4)/2 khỏi mỗi
  kênh, không tương đương ý nghĩa average-reference trên lưới nhiều kênh.
  Không nên diễn giải quá mức.
- **`segRej` (loại đoạn theo biên độ ±150µV)**: bật để có số liệu QC tường
  minh, nhưng trên thực nghiệm hầu như KHÔNG loại đoạn nào thêm — vì bước
  wavelet (chạy TRƯỚC segment) đã hạ biên độ mạnh sẵn, xem mục kết quả.

**Kết luận quan trọng cần biết trước khi đọc số liệu**: `Pct_Var_Retained_PostWavelet`
(% phương sai tín hiệu còn giữ lại sau wavelet) trên dữ liệu wearable 2 kênh
này THẤP HƠN NHIỀU so với các nghiên cứu HAPPE công bố trên EEG lưới dày
(thường 60-90%) — đây LÀ tín hiệu QC thật, phản ánh dữ liệu từ thiết
bị đeo trán/dry-electrode có tỉ lệ artefact/nhiễu cao hơn EEG lưới ướt lâm
sàng, KHÔNG nên so sánh trực tiếp con số % này với benchmark HAPPE gốc.

In [ ]:
from pathlib import Path
import re
import shutil
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

REPO_ROOT = Path("..").resolve()

LIB_DIR = REPO_ROOT / "lib"
HAPPE_SRC = REPO_ROOT / "happe-python" / "src"
for p in (LIB_DIR, HAPPE_SRC):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from happe import run_pipeline, default_params
import happe
# happe.qc.visualize gọi matplotlib.use("Agg") lúc import (cần cho HAPPE tự lưu
# PNG hàng loạt) -- việc này đổi backend GLOBAL cho cả kernel, làm mất hiển thị
# hình inline của chính notebook này (plt.show() sẽ không render gì cả dù
# fig.savefig() vẫn ghi file bình thường). Ép lại backend inline SAU khi import
# happe để các cell vẽ hình phía dưới hiển thị đúng trong notebook.
%matplotlib inline
print("happe:", happe.__file__, "| version:", happe.__version__)

## 1. Cấu hình HAPPE

In [ ]:
HAPPE_CONFIG = default_params(erp=False)  # resting/thiền, không phải ERP/task

HAPPE_CONFIG.lineNoise.freq = 50.0          # điện lưới Việt Nam
HAPPE_CONFIG.lineNoise.method = "notch"     # notch/band-stop thật (zero-phase), thay vì cleanline/spectrum_fit mặc định
HAPPE_CONFIG.filt.lowpass = 45.0            # băng thông 1-45Hz (thay vì 1-100Hz) theo yêu cầu
HAPPE_CONFIG.badChans.enabled = True
HAPPE_CONFIG.badChans.density = "low"       # xem giải thích ở mục giới thiệu -- 2 kênh, chỉ bắt flatline tin cậy
HAPPE_CONFIG.badChans.order = "before"      # phát hiện trước wavelet (mặc định HAPPE)
HAPPE_CONFIG.muscIL.enabled = False         # ICA không có ý nghĩa với 2 kênh -- xem giải thích ở trên
HAPPE_CONFIG.wavelet.method = "bayes"       # bước làm sạch artefact CHÍNH cho dữ liệu này
HAPPE_CONFIG.wavelet.threshold_rule = "soft" # DEVIATION -- soft thay vì hard mặc định của HAPPE cho resting: giữ lại
                                              # nhiều công suất alpha/beta/gamma hơn trên dữ liệu 2 kênh. preWaveletTddr
                                              # (xem config.py) giữ enabled=False (mặc định).
HAPPE_CONFIG.wavelet.block_seconds = 4.0     # DEVIATION -- BayesShrink theo block 4s thay vì toàn phiên: một artefact biên độ lớn
                                              # ở một đoạn không còn kéo ngưỡng của cả subband lên và xoá alpha thật ở đoạn sạch khác trong
                                              # cùng phiên .
HAPPE_CONFIG.segment.enabled = True
HAPPE_CONFIG.segment.reg_len_s = 2.0        # đoạn cố định 2s (resting paradigm)
HAPPE_CONFIG.segRej.enabled = True
HAPPE_CONFIG.segRej.by_amplitude = True
HAPPE_CONFIG.segRej.amp_min = -150e-6       # BUGFIX -- dữ liệu nội bộ ở đơn vị Volt (chuẩn MNE), không phải µV như
HAPPE_CONFIG.segRej.amp_max = 150e-6        # HAPPE MATLAB gốc; -150.0/150.0 (Volt) không bao giờ đạt tới nên loại epoch
                                              # theo biên độ đã VÔ HIỆU HOÁ từ đầu.
                                              # -150e-6/150e-6 (Volt) = đúng ±150µV như ý định ban đầu.
HAPPE_CONFIG.reref.enabled = True
HAPPE_CONFIG.reref.method = "average"       # xem lưu ý CHƯA THỂ XÁC NHẬN ở trên
HAPPE_CONFIG.vis.enabled = True             # xuất hình phổ công suất mỗi file
HAPPE_CONFIG.vis.freq_range_hz = (1.0, 45.0)  # khớp băng thông mới
HAPPE_CONFIG.outputFormat.formats = ["fif"]

HAPPE_CONFIG.validate()
print(HAPPE_CONFIG.to_yaml())

## 2. Gom file `eeg_labeled_raw.fif` từ Layer 2, dàn ra thư mục input phẳng

`run_pipeline()` quét 1 thư mục PHẲNG (không đệ quy) — copy từng file vào 1
thư mục staging với tên đã mã hoá sẵn participant/day/session (để join lại
định danh sau khi có báo cáo QC, không cần file mapping riêng). Dùng copy
(không symlink) để tránh vấn đề OneDrive cloud-only placeholder.

In [ ]:
LAYER2_DIR = REPO_ROOT / "data" / "analytics" / "layer2"
STAGING_IN_DIR = REPO_ROOT / "data" / "analytics" / "happe_qc" / "_staging_input"
OUT_DIR = REPO_ROOT / "data" / "analytics" / "happe_qc"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if STAGING_IN_DIR.exists():
    shutil.rmtree(STAGING_IN_DIR)
STAGING_IN_DIR.mkdir(parents=True)

PARTICIPANT_RE = re.compile(r"participant=([^/\\]+)")
STUDY_DAY_RE = re.compile(r"study_day=([^/\\]+)")
SESSION_RE = re.compile(r"session_id=([^/\\]+)")

fif_files = sorted(LAYER2_DIR.rglob("eeg_labeled_raw.fif"))
print(f"Tìm thấy {len(fif_files)} file eeg_labeled_raw.fif")

identity_by_stem = {}
for src in fif_files:
    s = str(src)
    participant = PARTICIPANT_RE.search(s).group(1)
    study_day = STUDY_DAY_RE.search(s).group(1)
    session_id = SESSION_RE.search(s).group(1)
    stem = f"{participant}_day{study_day}_{session_id[:8]}"
    dst = STAGING_IN_DIR / f"{stem}.fif"
    shutil.copyfile(src, dst)
    identity_by_stem[f"{stem}.fif"] = {
        "participant_key": participant, "study_day": int(study_day) if study_day.isdigit() else study_day,
        "session_id": session_id,
    }

identity_df = pd.DataFrame.from_dict(identity_by_stem, orient="index").rename_axis("File").reset_index()
print(f"Đã dàn {len(identity_by_stem)} file vào {STAGING_IN_DIR}")
identity_df.head()

## 3. Chạy HAPPE trên toàn bộ session

Mỗi file được xử lý trong try/except riêng (isolate lỗi, xem
`happe-python/src/happe/pipeline.py`) — 1 file lỗi không làm hỏng cả mẻ,
lỗi được ghi vào `HAPPE_errorLog.csv`.

In [ ]:
report = run_pipeline(HAPPE_CONFIG, input_dir=str(STAGING_IN_DIR), output_dir=str(OUT_DIR))

qc_dir = OUT_DIR / "7 - quality_assessment_outputs"
data_qc = pd.DataFrame(report.data_rows)
pipeline_qc = pd.DataFrame(report.pipeline_rows)
error_log = pd.DataFrame(report.error_rows)

print(f"Data QC: {len(data_qc)} dòng | Pipeline QC: {len(pipeline_qc)} dòng | Lỗi: {len(error_log)}")
if len(error_log):
    print(error_log.to_string(index=False))

## 4. Ghép định danh participant/day/session vào báo cáo QC

In [ ]:
data_qc = data_qc.merge(identity_df, on="File", how="left")
pipeline_qc = pipeline_qc.merge(identity_df, on="File", how="left")

cols_front = ["participant_key", "study_day", "session_id", "File"]
data_qc = data_qc[cols_front + [c for c in data_qc.columns if c not in cols_front]]
pipeline_qc = pipeline_qc[cols_front + [c for c in pipeline_qc.columns if c not in cols_front]]

data_qc_path = OUT_DIR / "happe_dataQC_with_identity.csv"
pipeline_qc_path = OUT_DIR / "happe_pipelineQC_with_identity.csv"
data_qc.to_csv(data_qc_path, index=False)
pipeline_qc.to_csv(pipeline_qc_path, index=False)
print("Đã lưu:", data_qc_path)
print("Đã lưu:", pipeline_qc_path)
data_qc.head()

## 5. Tổng quan số liệu QC

In [ ]:
summary_cols = ["File_Length_s", "Pct_Good_Channels", "Pct_Var_Retained_PostWavelet", "Pct_Epochs_Retained"]
print(data_qc[summary_cols].describe().to_string())

print("\nSố session theo participant:")
print(data_qc["participant_key"].value_counts().to_string())

## 6. Trực quan hoá

### 6.1. % phương sai còn lại sau wavelet — theo session, sắp xếp tăng dần
Chỉ số quan trọng nhất: phản ánh tỉ lệ tín hiệu HAPPE coi là "artefact/nhiễu"
bị loại ở bước làm sạch chính. Thấp = nhiều artefact, không nhất thiết = lỗi
pipeline (xem giải thích ở đầu notebook).

In [ ]:
plot_df = data_qc.sort_values("Pct_Var_Retained_PostWavelet").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab20(pd.factorize(plot_df["participant_key"])[0] % 20)
ax.bar(range(len(plot_df)), plot_df["Pct_Var_Retained_PostWavelet"], color=colors)
ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df["participant_key"] + "/d" + plot_df["study_day"].astype(str), rotation=90, fontsize=7)
ax.set_ylabel("% phương sai còn lại sau wavelet")
ax.set_title("Pct_Var_Retained_PostWavelet theo session (mỗi màu = 1 participant)")
ax.axhline(plot_df["Pct_Var_Retained_PostWavelet"].median(), color="red", linestyle="--", linewidth=1, label=f"median={plot_df['Pct_Var_Retained_PostWavelet'].median():.1f}%")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "01_pct_var_retained_by_session.png", dpi=130)
plt.show()

### 6.2. Phân bố theo participant (boxplot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
participants = sorted(data_qc["participant_key"].unique())
box_data = [data_qc.loc[data_qc["participant_key"] == p, "Pct_Var_Retained_PostWavelet"].dropna() for p in participants]
ax.boxplot(box_data, tick_labels=participants, showmeans=True)
ax.set_ylabel("% phương sai còn lại sau wavelet")
ax.set_title("Phân bố theo participant")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig(OUT_DIR / "02_pct_var_retained_by_participant.png", dpi=130)
plt.show()

### 6.3. Tần số điện lưới bị loại đúng chỗ? (coherence tại 50Hz vs lân cận)
Coherence THẤP tại đúng 50Hz/100Hz (so với các tần số lân cận gần 1.0) xác
nhận line-noise reduction nhắm đúng tần số điện lưới (50Hz).

In [ ]:
coh_cols = [c for c in pipeline_qc.columns if c.startswith("lineNoise_coherence_")]
coh_freqs = sorted([float(c.replace("lineNoise_coherence_", "").replace("Hz", "")) for c in coh_cols])
coh_means = [pipeline_qc[f"lineNoise_coherence_{f:g}Hz"].mean() for f in coh_freqs]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(coh_freqs, coh_means, marker="o")
for f in (50, 100):
    if f in coh_freqs:
        ax.axvline(f, color="red", linestyle=":", linewidth=1)
ax.set_xlabel("Tần số (Hz)")
ax.set_ylabel("Coherence trung bình (pre vs post line-noise reduction)")
ax.set_title("Coherence theo tần số -- thấp tại 50/100Hz = lọc đúng điện lưới VN")
fig.tight_layout()
fig.savefig(OUT_DIR / "03_line_noise_coherence_by_freq.png", dpi=130)
plt.show()

### 6.4. % kênh tốt & % đoạn giữ lại (nhắc lại: n=2 kênh, diễn giải thận trọng)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(data_qc["Pct_Good_Channels"].dropna(), bins=10, color="#0891b2", edgecolor="white")
axes[0].set_title("Pct_Good_Channels (n=2 kênh -- xem giới hạn ở mục giới thiệu)")
axes[0].set_xlabel("%")
axes[1].hist(data_qc["Pct_Epochs_Retained"].dropna(), bins=10, color="#7c3aed", edgecolor="white")
axes[1].set_title("Pct_Epochs_Retained (loại đoạn theo biên độ ±150µV)")
axes[1].set_xlabel("%")
fig.tight_layout()
fig.savefig(OUT_DIR / "04_good_channels_epochs_retained.png", dpi=130)
plt.show()

### 6.5. Ví dụ hình phổ công suất do HAPPE xuất (1 session)
HAPPE tự vẽ 1 hình `*_vis.png` (phổ công suất, bước 21) cho mỗi file, lưu ở
`7 - quality_assessment_outputs/`. Hiện 1 ví dụ để kiểm tra trực quan.

In [ ]:
qc_out_dir = OUT_DIR / "7 - quality_assessment_outputs"
example_vis = sorted(qc_out_dir.glob("*_vis.png"))
if example_vis:
    from matplotlib import image as mpimg
    img = mpimg.imread(example_vis[0])
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(example_vis[0].name)
    plt.show()
else:
    print("Không tìm thấy hình *_vis.png -- kiểm tra lại params.vis.enabled")

## 7. Bảng tổng hợp cuối + đường dẫn output

Toàn bộ cấu trúc thư mục output theo đúng chuẩn HAPPE (xem README):
`1 - intermediate_processing`, `2 - wavelet_cleaned_continuous`,
`5 - segmenting`, `6 - processed` (file `.fif` đã xử lý),
`7 - quality_assessment_outputs` (3 CSV QC gốc của HAPPE + `run_config.yaml`
để tái lập đúng tham số đã dùng).

In [ ]:
print("Output directory:", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name)

print("\n=== Bảng tổng hợp (participant/day/session x các chỉ số QC chính) ===")
data_qc[["participant_key", "study_day", "session_id", "File_Length_s",
         "Pct_Good_Channels", "Pct_Var_Retained_PostWavelet", "Pct_Epochs_Retained"]].sort_values(
    ["participant_key", "study_day"]
).reset_index(drop=True)